# Data Collection

## [**Web scraping**](https://realpython.com/beautiful-soup-web-scraper-python/) 


Web scraping is the process of (automated) data collection from the web. When web scraping, it is important to be informed about potential [restrictions and prohibitions](https://benbernardblog.com/web-scraping-and-crawling-are-perfectly-legal-right/). Benoit Bernard highlights several key steps to ensure we do not violate rights and provisions related to the data we collect:

1. Use an API if available
2. Respect the Terms of Service
3. Respect [*robots.txt*](https://www.inf.uniri.hr/robots.txt)
4. Use a reasonable *crawl rate* – do not overload with requests 
5. ...

> *"You write your code once, and it will get the information you want many times and from many pages."*

### Exploring the Website

To effectively automate the data collection process, it is necessary to get closely acquainted with the structure of the website itself. In these exercises, our task is to download all course syllabi from the [**Faculty of Informatics and Digital Technologies**](https://www.inf.uniri.hr/). By interacting with the website, find the page where the course syllabus information is located.

#### URL Analysis

> In the meantime, as expected, the FIDIT website changed its structure, so we can use the web crawler/archive [Wayback Machine](https://web.archive.org/), which builds a digital library and Internet archive, to access the archived version of the web page.

> As a [prefix](https://en.wikipedia.org/wiki/Prefix) to each URL, let's prepend `https://web.archive.org/web/20240414200035/`.

Individual URLs reveal useful information about the website itself, e.g., from the URL:     

https://www.inf.uniri.hr/nastava/izvedbeni-programi -> https://web.archive.org/web/20240414200035/https://www.inf.uniri.hr/nastava/izvedbeni-programi

it is possible to infer the URL containing undergraduate study course syllabi:  
 
https://www.inf.uniri.hr/nastava/izvedbeni-programi/prijediplomski-studij. 
 
Similarly, we can deduce that the graduate study course syllabi are located at:      
 
https://www.inf.uniri.hr/nastava/izvedbeni-programi/diplomski-studij. 
 
We should not jump to conclusions; although a structure exists, it is **most often** not consistent.
 
![](./Images/404_kategorija_nije_pronadena.png)

#### Website Analysis with Developer Tools

Developer tools allow us to directly edit pages, diagnose issues, and inspect the structure.


![](./Images/Dev_tools.png)



### Fetching Data from the Website

Using the [requests](https://www.w3schools.com/python/module_requests.asp) library, let's fetch the page containing the undergraduate study syllabi: https://www.inf.uniri.hr/nastava/izvedbeni-programi/prijediplomski-studij

> Comment: *During class, we used the Wayback Machine to access the web page for which these exercises were prepared, for example: https://web.archive.org/web/20240414200035/https://www.inf.uniri.hr/nastava/izvedbeni-programi/prijediplomski-studij.*
>
> *Let's prepend the prefix `WBM = "https://web.archive.org/web/20240414200035/"` to each link*.

In [ ]:
import requests 

WBM = "https://web.archive.org/web/20240414200035/" # Wayback Machine
URL = "https://www.inf.uniri.hr/nastava/izvedbeni-programi/prijediplomski-studij"
page = requests.get(WBM + URL) # Added prefix

print(page.text)

`print(page.text)` returns the HTML content we fetched:
```shell
Your request is being blocked because our system has flagged it as suspected abusive bot traffic that is degrading the performance of the Wayback Machine.
<p>
We recognize this might be a false positive designation, that you might be a normal human and browser just trying to use our service.  We apologize if that is the case.
<p>
Please know we are actively working to provide better service to humans and block high-volume robotic traffic that is overloading our systems.  We wish that was an easier task.  
<p>
Please email info@archive.org with your OS/Browser and IP number if you would like us to look into this.
```

We are not satisfied with this response. Let's introduce ourselves to the server—perhaps it will return more content:

Let's add headers:
```python
headers = {
    'User-Agent': 'UTOR Course Agent 2026',
    'From': 'andrija.poleksic@uniri.hr'  # This is another valid field
}
```


In [ ]:
WBM = "https://web.archive.org/web/20240414200035/" # Wayback Machine
URL = "https://www.inf.uniri.hr/nastava/izvedbeni-programi/prijediplomski-studij"
headers = {
    'User-Agent': 'UTOR Course Agent 2026',
    'From': 'andrija.poleksic@uniri.hr'  # This is another valid field
}

page = requests.get(WBM + URL, headers=headers) # Added prefix

print(page.text)

By printing the content of the `text` attribute of the `page` object, we see HTML content identical to what is shown in the *inspector* window of developer tools. Great news for us: the part of the website we are interested in is static! Let's try to extract interesting information—for example, all course titles and their lecturers.

#### Extracting Data with split + regex

> TASK 1. Extract course title and lecturer data using the `split()` method

In [ ]:
# Using the split method

retci_tablice = page.text.split("<tr>")
print(retci_tablice)

for redak in retci_tablice:
    try:
        print(redak.split("<td>")[2].replace("</td>", ""))
        print(redak.split("<td>")[3].replace("</td>", ""))
        print("-"*20)
    except IndexError:
        print("")

> TASK 2. Extract links using the `split()` method

In [ ]:
for redak in retci_tablice:
    try:
        print(redak.split("<td>")[4])
    except IndexError:
        print("")

For more complex examples, the `split()` method quickly becomes inefficient. Let's extract the links using regular expressions.

> TASK 3. Extract links using the [re](https://docs.python.org/3/library/re.html) library; example: 

`<a href="/images/nastava/izvedbeni/2023_2024/PDS/1/DINP_FIDIT_2023_2024_OPROG.pdf" target="_blank"><i class="fa fa-file-pdf-o"></i></a></td>`



Methods `re.search()`, `re.match()`, `re.fullmatch()`:
- [re.match()](https://docs.python.org/3/library/re.html#re.match) checks for a match only at the beginning of the string
- [re.search()](https://docs.python.org/3/library/re.html#re.search) checks for a match anywhere in the string (this is what Perl does by default)
- [re.fullmatch()](https://docs.python.org/3/library/re.html#re.fullmatch) checks for entire string to be a match

- [re.compile()](https://docs.python.org/3/library/re.html#re.compile) compiles regular expression


In [ ]:
import re

# Example on a single string
string = "<a href=\"/images/nastava/izvedbeni/2023_2024/PDS/1/DINP_FIDIT_2023_2024_OPROG.pdf\" target=\"_blank\"><i class=\"fa fa-file-pdf-o\"></i></a></td>"
regex = re.compile(r"href=\"([\w\/-_\.]*)\"")
                     
print(regex)
print(re.search(regex, string))



In [ ]:
# Example across all table rows
linkovi = []
for redak in retci_tablice:
    try:
        linkovi.append(re.search(regex, redak).group(1))
        print(re.search(regex, redak).group(1))
    except AttributeError:
        print("")

In [ ]:
print(linkovi)

#### Extracting Data Using the BeautifulSoup Library

> TASK 4. Extract course title and lecturer data using HTML tags

[BeautifulSoup4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) is a Python library for pulling data out of HTML/XML files. BeautifulSoup supports numerous [parsers](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#installing-a-parser):

| Parser | Typical usage | Advantages | Disadvantages |
| ------ | ------------- | ---------- | ------------- |
| Python’s html.parser | `BeautifulSoup(markup, "html.parser")`| - Batteries included <br> - Decent speed | - Not as fast as lxml, less lenient than html5lib. |
| [**lxml’s HTML parser**](https://lxml.de/) | `BeautifulSoup(markup, "lxml")` | - Very fast | - External C dependency |
| lxml’s XML parser | `BeautifulSoup(markup, "lxml-xml") BeautifulSoup(markup, "xml")` | - Very fast <br> - The only currently supported XML parser | - External C dependency |
| html5lib | `BeautifulSoup(markup, "html5lib")` | - Extremely lenient <br> - Parses pages the same way a web browser does <br> - Creates valid HTML5 | - Very slow <br> - External Python dependency |





##### Basic Navigation Using BS4

To start, let's fetch the page with the graduate study syllabi: `https://www.inf.uniri.hr/izvedbeni-programi/diplomski-studij-informatike`

In [ ]:
import requests

URL_DIP = "https://www.inf.uniri.hr/izvedbeni-programi/diplomski-studij-informatike"

page_dip = requests.get(WBM + URL_DIP, headers=headers) # Added prefix

print(page_dip.text)

Let's create a new soup object using the BeautifulSoup constructor:

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(page_dip.text, 'html.parser')

print(soup.prettify())

In [ ]:
# Content under the "title" HTML tag
print(soup.title)

# Name of the HTML tag
print(soup.title.name)

# Content of the HTML tag
print(soup.title.string)

# Name of the "parent" HTML element
print(soup.title.parent.name)

# First element in the tree with the p tag - paragraph
print(soup.p)

# span element inside the first paragraph element
print(soup.p.span)

# Style of the span element
print(soup.p.span["style"])

# First element with the a - anchor tag
print(soup.a)

# All elements with the span tag
print(soup.find_all('span'))

# First element with the given id parameter
soup.find(id="copyright-year")

> TASK 5. Find the element whose class is "item-294".

In [ ]:
# class is a reserved keyword in Python for class definitions!
print(soup.find(class_="item-294"))

> TASK 6. Find all elements whose class is "item-N", where N is a three-digit number


In [ ]:
import re
print(soup.find_all(class_=re.compile(r"item-[0-9]{3}")))

> ZADATAK 3* 